In [1]:
import os

# 1. CONNECT TO GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

# 2. TELL COLAB WHERE YOUR KEY IS
# This points to the folder you just created in Phase 1
os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/My Drive/Kaggle"

# 3. DOWNLOAD KDD CUP 2015 DATA (AS A DATASET)
print("Downloading Data...")
# Use the -d flag for datasets instead of -c for competitions
!kaggle datasets download -d sst2023/kdd-cup-2015

# 4. UNZIP THE FILES
print("Unzipping...")
!unzip -q kdd-cup-2015.zip

# 5. LIST FILES (To confirm it worked)
print("Done! Here are your files:")
!ls

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/sst2023/kdd-cup-2015
License(s): unknown
100% 119M/119M [00:01<00:00, 115MB/s]

Unzipping...
Done! Here are your files:
date.csv  drive  kdd-cup-2015.zip  object.csv  sample_data  test  train


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import os

print("Loading raw data...")
log_train = pd.read_csv("train/train/log_train.csv")
log_test = pd.read_csv("test/test/log_test.csv")
object_df = pd.read_csv("object.csv")
truth_train = pd.read_csv("train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv("test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Loading raw data...


In [3]:
# ==========================================
# FILTER FIRST 2 WEEKS  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

print(f"log_train (first 2 weeks): {log_train.shape}, log_test: {log_test.shape}")

log_train (first 2 weeks): (6004564, 5), log_test: (3981037, 5)


In [4]:
# ==========================================
# MERGE CATEGORY  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

In [5]:
# =====================================================================
# FSLSM FEATURE ENGINEERING — all 4 dimensions, rule-based S-score
# (per-dimension proportional split, no clustering — see conversation
# for the fixes applied here vs. the original version):
#
#   - Processing: dwell time now measured FORWARD (time to next event,
#     not gap since previous), capped at a session timeout so cross-
#     session breaks aren't counted as "reflection". Active (count) and
#     Reflective (seconds) are compared via percentile rank, not raw
#     magnitude, since a count and a sum-of-seconds are never on
#     comparable scales.
#   - Perception: Sensing (retries) and Intuitive (wiki+breadth) also
#     compared via percentile rank, since navigate breadth structurally
#     runs larger than retries regardless of individual preference.
#   - Understanding: jump detection uses object.csv's module hierarchy
#     ('children' column) when available -- a navigate transition to a
#     module that isn't the previous module's parent/child/sibling is a
#     genuine "jump between non-adjacent modules", matching the FSLSM
#     mapping table literally. Falls back to the old time-gap-outlier
#     heuristic (renamed and clearly labeled as a proxy) only if
#     object.csv has no usable hierarchy. page_close removed from the
#     Global score -- it wasn't part of the original mapping table.
# =====================================================================

SESSION_TIMEOUT_SEC = 1800  # 30 min; longer gaps are treated as time away, not dwell on the item

def add_raw_time_gap(df):
    # Dwell time attributed to an event = time to the NEXT event by the same
    # learner (forward gap), not the gap since the previous one — otherwise
    # "reflective dwell on this video" is actually measuring idle time
    # BEFORE the video started. Capped at SESSION_TIMEOUT_SEC so a break
    # between sessions (hours/days) doesn't get misattributed as reflection.
    df = df.sort_values(['enrollment_id', 'time']).copy()
    fwd_gap = (df.groupby('enrollment_id')['time'].shift(-1) - df['time']).dt.total_seconds()
    df['time_gap_sec'] = fwd_gap.fillna(0).clip(upper=SESSION_TIMEOUT_SEC)
    return df


log_train = add_raw_time_gap(log_train)
log_test = add_raw_time_gap(log_test)


def compute_input_dimension(df):
    # V_i = video_i + access_i ; R_i = wiki_i + discussion_i  (verbatim from the PDF)
    counts = df.groupby(['enrollment_id', 'event']).size().unstack(fill_value=0)
    for col in ['video', 'access', 'wiki', 'discussion']:
        if col not in counts.columns:
            counts[col] = 0
    V = counts['video'] + counts['access']
    R = counts['wiki'] + counts['discussion']
    total = V + R
    S = np.where(total > 0, (V - R) / total, 0.0)
    label = np.where(V > R, 1, 0)  # 1 = Visual, 0 = Verbal (ties -> Verbal)
    return pd.DataFrame({'student_id': counts.index, 'input_S': S, 'input_label': label}).reset_index(drop=True)


def compute_processing_dimension(df):
    # Active_i = count(problem) + count(discussion) — literal count.
    active_counts = df[df['event'].isin(['problem', 'discussion'])].groupby('enrollment_id').size()
    # Reflective_i = summed FORWARD dwell time (time_gap_sec, session-capped) on video/access events.
    reflective_dwell = df[df['event'].isin(['video', 'access'])].groupby('enrollment_id')['time_gap_sec'].sum()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['A'] = active_counts
    out['Rf'] = reflective_dwell
    out = out.fillna(0.0)

    # Percentile-rank normalize before comparing: A is a count, Rf is summed
    # seconds — comparing raw magnitudes lets whichever has bigger natural
    # units dominate regardless of actual preference. method='min' keeps
    # tied-lowest values (e.g. the large block of A=0 or Rf=0 students) at
    # the bottom of the ranking instead of the middle.
    A_norm = out['A'].rank(method='min', pct=True)
    Rf_norm = out['Rf'].rank(method='min', pct=True)
    total = A_norm + Rf_norm
    S = np.where(total > 0, (A_norm - Rf_norm) / total, 0.0)
    label = np.where(A_norm > Rf_norm, 1, 0)  # 1 = Active, 0 = Reflective (ties -> Reflective)
    out = out.reset_index()
    out['processing_S'] = S
    out['processing_label'] = label
    return out[['student_id', 'processing_S', 'processing_label']]


def compute_perception_dimension(df):
    # Sensing_i: repeat visits to the SAME object during 'problem' events, beyond the first (= retries).
    problem_df = df[df['event'] == 'problem']
    visits_per_object = problem_df.groupby(['enrollment_id', 'object']).size()
    retries_per_object = (visits_per_object - 1).clip(lower=0)
    sensing_retries = retries_per_object.groupby('enrollment_id').sum()
    # Intuitive_i: count(wiki) + breadth of navigate exploration (# distinct objects reached via navigate).
    wiki_counts = df[df['event'] == 'wiki'].groupby('enrollment_id').size()
    navigate_breadth = df[df['event'] == 'navigate'].groupby('enrollment_id')['object'].nunique()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['Sn'] = sensing_retries
    out['wiki'] = wiki_counts
    out['breadth'] = navigate_breadth
    out = out.fillna(0.0)
    out['In'] = out['wiki'] + out['breadth']

    # Percentile-rank normalize before comparing: 'breadth' structurally runs
    # larger than 'retries' regardless of individual preference (more places
    # to navigate to than problems to retry), so compare each learner's RANK
    # on each axis rather than raw magnitudes.
    Sn_norm = out['Sn'].rank(method='min', pct=True)
    In_norm = out['In'].rank(method='min', pct=True)
    total = Sn_norm + In_norm
    S = np.where(total > 0, (Sn_norm - In_norm) / total, 0.0)
    label = np.where(Sn_norm > In_norm, 1, 0)  # 1 = Sensing, 0 = Intuitive (ties -> Intuitive)
    out = out.reset_index()
    out['perception_S'] = S
    out['perception_label'] = label
    return out[['student_id', 'perception_S', 'perception_label']]


def _count_jumps(gaps):
    # FALLBACK ONLY: time-gap-outlier heuristic, used when object.csv has no
    # usable module hierarchy. This is NOT "jump between non-adjacent
    # modules" — it's a proxy based on pause length between navigate events,
    # which can also reflect the learner stepping away, not jumping modules.
    gaps = np.asarray(gaps)
    if len(gaps) < 4:
        return 0
    q1, q3 = np.percentile(gaps, [25, 75])
    threshold = q3 + 1.5 * (q3 - q1)
    return int((gaps > threshold).sum())


def build_adjacency_map(object_df):
    # Builds each module's structural neighbors (parent, children, siblings)
    # from object.csv's 'children' column (space-separated child module_ids).
    # Returns None if the column isn't usable, so callers can fall back.
    if 'children' not in object_df.columns:
        return None

    parent_of = {}
    children_of = {}
    for _, row in object_df.iterrows():
        mid = row['module_id']
        children_raw = row.get('children', None)
        if pd.isna(children_raw) or str(children_raw).strip() == '':
            continue
        child_ids = str(children_raw).split()
        children_of[mid] = child_ids
        for c in child_ids:
            parent_of[c] = mid

    if not children_of:
        return None  # column existed but had no parseable hierarchy in it

    neighbors = {}
    for m in set(object_df['module_id']):
        neigh = set()
        p = parent_of.get(m)
        if p is not None:
            neigh.add(p)
            neigh.update(children_of.get(p, []))
        neigh.update(children_of.get(m, []))
        neigh.discard(m)
        neighbors[m] = neigh
    return neighbors


# Built once, shared by train/test so "adjacent" means the same thing for both.
neighbors_map = build_adjacency_map(object_df)
JUMP_METHOD = "structural (object.csv hierarchy)" if neighbors_map else "temporal_irregularity (fallback proxy, NOT structural)"
print(f"Understanding dimension jump detection: {JUMP_METHOD}")


def compute_understanding_dimension(df, neighbors=None):
    nav_df = df[df['event'] == 'navigate'].sort_values(['enrollment_id', 'time']).copy()
    nav_counts = nav_df.groupby('enrollment_id').size()

    if neighbors is not None:
        # STRUCTURAL jump: navigate transition to a module that is not the
        # previous module's parent, child, or sibling. This is a literal
        # implementation of "jump between non-adjacent modules".
        prev_obj = nav_df.groupby('enrollment_id')['object'].shift(1)
        curr_obj = nav_df['object']
        is_jump = [
            (not pd.isna(p)) and (c != p) and (c not in neighbors.get(p, set()))
            for p, c in zip(prev_obj, curr_obj)
        ]
        nav_df['is_jump'] = is_jump
        jump_counts = nav_df.groupby('enrollment_id')['is_jump'].sum()
    else:
        # FALLBACK: no usable hierarchy in object.csv, so this uses the
        # time-gap-outlier proxy instead (see _count_jumps docstring above).
        nav_df['nav_gap'] = nav_df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0)
        jump_counts = nav_df.groupby('enrollment_id')['nav_gap'].apply(_count_jumps)

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['nav_total'] = nav_counts
    out['jumps'] = jump_counts
    out = out.fillna(0.0)
    out['Sq'] = (out['nav_total'] - out['jumps']).clip(lower=0)
    out['Gl'] = out['jumps']  # page_close removed — not part of the original FSLSM mapping table
    total = out['Sq'] + out['Gl']
    S = np.where(total > 0, (out['Sq'] - out['Gl']) / total, 0.0)
    label = np.where(out['Sq'] > out['Gl'], 1, 0)  # 1 = Sequential, 0 = Global (ties -> Global)
    out = out.reset_index()
    out['understanding_S'] = S
    out['understanding_label'] = label
    return out[['student_id', 'understanding_S', 'understanding_label']]


def compute_all_fslsm_dimensions(df, neighbors=None):
    out = compute_input_dimension(df) \
        .merge(compute_processing_dimension(df), on='student_id', how='outer') \
        .merge(compute_perception_dimension(df), on='student_id', how='outer') \
        .merge(compute_understanding_dimension(df, neighbors=neighbors), on='student_id', how='outer')
    return out.fillna(0.0)

fslsm_train = compute_all_fslsm_dimensions(log_train, neighbors=neighbors_map)
fslsm_test = compute_all_fslsm_dimensions(log_test, neighbors=neighbors_map)

print("\nFSLSM feature summary (train):")
print(fslsm_train.describe())
for dim in ['input', 'processing', 'perception', 'understanding']:
    print(f"  {dim}_label distribution: {fslsm_train[f'{dim}_label'].value_counts().to_dict()}")

os.makedirs("data/processed", exist_ok=True)
fslsm_train.to_csv("data/processed/fslsm_full_train.csv", index=False)
fslsm_test.to_csv("data/processed/fslsm_full_test.csv", index=False)
print("\nSaved data/processed/fslsm_full_train.csv / fslsm_full_test.csv (for Student 2 & 3 to reuse).")


Understanding dimension jump detection: structural (object.csv hierarchy)

FSLSM feature summary (train):
          student_id        input_S    input_label   processing_S  \
count  120542.000000  120542.000000  120542.000000  120542.000000   
mean   100364.330748       0.584440       0.718198      -0.272391   
std     58003.738782       0.472692       0.449879       0.502676   
min         1.000000      -1.000000       0.000000      -0.999983   
25%     50060.250000       0.000000       0.000000      -0.999950   
50%    100360.500000       0.813953       1.000000       0.000000   
75%    150658.500000       1.000000       1.000000       0.016520   
max    200905.000000       1.000000       1.000000       0.999983   

       processing_label   perception_S  perception_label  understanding_S  \
count     120542.000000  120542.000000     120542.000000    120542.000000   
mean           0.284092      -0.609639          0.227564         0.185302   
std            0.450983       0.582949   

In [6]:
# ==========================================
# FEATURE ENGINEERING  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)

print(f"train_df: {train_df.shape}, test_df: {test_df.shape}")

train_df: (120542, 4), test_df: (80362, 4)


In [7]:
# ==========================================
# MERGE ALL FOUR STATIC FSLSM DIMENSIONS ONTO THE SEQUENCE TABLES
# ==========================================
FSLSM_COLS = ['input_S', 'input_label', 'processing_S', 'processing_label',
              'perception_S', 'perception_label', 'understanding_S', 'understanding_label']

train_df = train_df.merge(fslsm_train, on="student_id", how="left")
test_df = test_df.merge(fslsm_test, on="student_id", how="left")

missing_train = train_df[FSLSM_COLS].isna().any(axis=1).sum()
missing_test = test_df[FSLSM_COLS].isna().any(axis=1).sum()
print(f"Rows missing FSLSM features -> train: {missing_train}, test: {missing_test}")
# Every student here comes from the same filtered logs used to build fslsm_train/test,
# so this should be 0; fillna(0) is a safety net.
train_df[FSLSM_COLS] = train_df[FSLSM_COLS].fillna(0)
test_df[FSLSM_COLS] = test_df[FSLSM_COLS].fillna(0)

Rows missing FSLSM features -> train: 0, test: 0


In [8]:
# ==========================================
# ENCODING (source / event / category streams) — identical to 02_Data_Preprocessing.ipynb
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat
    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)

In [9]:
# ==========================================
# STATIC FEATURE VECTOR: 4 continuous S-scores (already in [-1, 1], no scaling needed)
# + 4 binary labels  ->  8-dim static vector per learner
# ==========================================
train_static = train_df[FSLSM_COLS].values.astype(np.float32)
test_static = test_df[FSLSM_COLS].values.astype(np.float32)

STATIC_DIM = train_static.shape[1]
print(f"Static feature vector per learner ({STATIC_DIM}-dim): {FSLSM_COLS}")

Static feature vector per learner (8-dim): ['input_S', 'input_label', 'processing_S', 'processing_label', 'perception_S', 'perception_label', 'understanding_S', 'understanding_label']


In [10]:
# ==========================================
# VARIABLE-LENGTH SEQUENCES + aligned static feature vectors
# ==========================================
def build_variable_length(df, static_arr):
    src_list = [torch.LongTensor(s) for s in df["src"]]
    evt_list = [torch.LongTensor(e) for e in df["evt"]]
    cat_list = [torch.LongTensor(c) for c in df["cat"]]
    gap_list = [torch.FloatTensor(g) for g in df["gaps"]]
    static_list = [torch.FloatTensor(static_arr[i]) for i in range(len(df))]
    y = torch.FloatTensor(df["dropout"].values)
    ids = df["student_id"].values
    return src_list, evt_list, cat_list, gap_list, static_list, y, ids

print("Building variable-length sequences (sequential streams + 8-dim FSLSM static vector)...")
src_train, evt_train, cat_train, gap_train, static_train_t, y_train, train_ids = build_variable_length(train_df, train_static)
src_test, evt_test, cat_test, gap_test, static_test_t, y_test, test_ids = build_variable_length(test_df, test_static)

lengths_train = [len(s) for s in src_train]
print(f"Train sequence lengths -> min: {min(lengths_train)}, max: {max(lengths_train)}, mean: {np.mean(lengths_train):.1f}")

Building variable-length sequences (sequential streams + 8-dim FSLSM static vector)...
Train sequence lengths -> min: 1, max: 3739, mean: 50.0


In [11]:
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, static, y=None):
        self.src, self.evt, self.cat, self.gaps, self.static = src, evt, cat, gaps, static
        self.y = y

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        if self.y is None:
            return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx]
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx], self.y[idx]


def collate_fn(batch):
    has_labels = len(batch[0]) == 6
    if has_labels:
        src, evt, cat, gaps, static, y = zip(*batch)
    else:
        src, evt, cat, gaps, static = zip(*batch)

    lengths = torch.LongTensor([len(s) for s in src])

    src = pad_sequence(src, batch_first=True, padding_value=0)
    evt = pad_sequence(evt, batch_first=True, padding_value=0)
    cat = pad_sequence(cat, batch_first=True, padding_value=0)
    gaps = pad_sequence(gaps, batch_first=True, padding_value=0.0)
    static = torch.stack(static)  # fixed-length, no padding needed

    if has_labels:
        y = torch.stack(y)
        return src, evt, cat, gaps, static, lengths, y
    return src, evt, cat, gaps, static, lengths


class BucketBatchSampler(Sampler):
    # Identical to 03_Model_Training_Evaluation.ipynb: groups similar-length
    # sequences into the same batch, then shuffles batch ORDER (not batch contents).
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = [indices[i:i+self.batch_size].tolist() for i in range(0, len(indices), self.batch_size)]
        if self.shuffle:
            np.random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


BATCH_SIZE = 128
train_lengths = [len(s) for s in src_train]
test_lengths = [len(s) for s in src_test]

train_sampler = BucketBatchSampler(train_lengths, batch_size=BATCH_SIZE, shuffle=True)
test_sampler = BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False)

train_loader = DataLoader(SeqDataset(src_train, evt_train, cat_train, gap_train, static_train_t, y_train),
                           batch_sampler=train_sampler, collate_fn=collate_fn)
test_loader = DataLoader(SeqDataset(src_test, evt_test, cat_test, gap_test, static_test_t, y_test),
                          batch_sampler=test_sampler, collate_fn=collate_fn)

In [12]:
class BiLSTMModel(nn.Module):
    # Sequential-only baseline (identical to 03_Model_Training_Evaluation.ipynb).
    def __init__(self):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)
        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, src, evt, cat, gaps, lengths, static=None):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)
        return torch.sigmoid(self.fc(h_cat)).squeeze()


class BiLSTM_FSLSM_Model(nn.Module):
    # Sequential BiLSTM backbone (same as baseline) fused with the 8-dim all-dimension
    # static FSLSM branch via concatenation before the classifier head.
    def __init__(self, static_dim):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)
        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)

        self.static_mlp = nn.Sequential(
            nn.Linear(static_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )
        self.fc = nn.Linear(128 + 16, 1)

    def forward(self, src, evt, cat, gaps, lengths, static):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)

        static_emb = self.static_mlp(static)           # (batch, 16)
        fused = torch.cat([h_cat, static_emb], dim=1)  # (batch, 144)
        return torch.sigmoid(self.fc(fused)).squeeze()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

Training on device: cuda


In [13]:
def train_and_evaluate(model, model_name, uses_static, epochs=3, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, evt, cat, gaps, static, lengths, labels in train_loader:
            src, evt, cat, gaps, static, labels = [t.to(device) for t in (src, evt, cat, gaps, static, labels)]
            optimizer.zero_grad()
            outputs = model(src, evt, cat, gaps, lengths, static) if uses_static else model(src, evt, cat, gaps, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[{model_name}] Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    preds = []
    with torch.no_grad():
        for src, evt, cat, gaps, static, lengths, _ in test_loader:
            src, evt, cat, gaps, static = [t.to(device) for t in (src, evt, cat, gaps, static)]
            outputs = model(src, evt, cat, gaps, lengths, static) if uses_static else model(src, evt, cat, gaps, lengths)
            preds.extend(outputs.cpu().numpy())

    ordered_ids = []
    for batch in BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False):
        ordered_ids.extend([test_ids[i] for i in batch])

    eval_df = pd.DataFrame({"student_id": ordered_ids, "pred": preds}).merge(truth_test, on="student_id")
    eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Precision": precision_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Recall": recall_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "F1": f1_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "AUC": roc_auc_score(eval_df["dropout"], eval_df["pred"]),
    }
    print(f"\n[{model_name}] Test metrics:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"  {k:<10}: {v:.4f}")
    return metrics, eval_df

In [14]:
torch.manual_seed(SEED)
np.random.seed(SEED)  # also reset numpy so BucketBatchSampler shuffle order matches Cell 14's run
baseline_model = BiLSTMModel()
baseline_metrics, baseline_eval_df = train_and_evaluate(
    baseline_model, model_name="Baseline BiLSTM (sequential only)", uses_static=False
)

[Baseline BiLSTM (sequential only)] Epoch 1 Loss: 0.4492
[Baseline BiLSTM (sequential only)] Epoch 2 Loss: 0.3927
[Baseline BiLSTM (sequential only)] Epoch 3 Loss: 0.3754

[Baseline BiLSTM (sequential only)] Test metrics:
  Accuracy  : 0.8447
  Precision : 0.8748
  Recall    : 0.9381
  F1        : 0.9053
  AUC       : 0.8323


In [15]:
torch.manual_seed(SEED)
np.random.seed(SEED)  # also reset numpy so BucketBatchSampler shuffle order matches Cell 13's run
fslsm_model = BiLSTM_FSLSM_Model(static_dim=STATIC_DIM)
fslsm_metrics, fslsm_eval_df = train_and_evaluate(
    fslsm_model, model_name="BiLSTM + FSLSM all-dimension static features (concat fusion)", uses_static=True
)

[BiLSTM + FSLSM all-dimension static features (concat fusion)] Epoch 1 Loss: 0.4395
[BiLSTM + FSLSM all-dimension static features (concat fusion)] Epoch 2 Loss: 0.3960
[BiLSTM + FSLSM all-dimension static features (concat fusion)] Epoch 3 Loss: 0.3751

[BiLSTM + FSLSM all-dimension static features (concat fusion)] Test metrics:
  Accuracy  : 0.8423
  Precision : 0.8673
  Recall    : 0.9456
  F1        : 0.9048
  AUC       : 0.8320


In [16]:
comparison_df = pd.DataFrame([baseline_metrics, fslsm_metrics])
comparison_df["Fusion Approach"] = ["N/A (no static features)", "Concatenation (intermediate fusion)"]
comparison_df["Input Features"] = [
    "Sequential (source/event/category tokens + time gaps)",
    "Sequential + static FSLSM all-dimension features (Input, Processing, Perception, "
    "Understanding: S-score + binary label each, 8-dim total)"
]
comparison_df = comparison_df[["Model", "Fusion Approach", "Input Features", "Accuracy", "Precision", "Recall", "F1", "AUC"]]

print(comparison_df.to_string(index=False))

improvement = fslsm_metrics["AUC"] - baseline_metrics["AUC"]
print(f"\nAUC change from adding all four FSLSM static dimensions: {improvement:+.4f}")

os.makedirs("data/results", exist_ok=True)
comparison_df.to_csv("data/results/student1_lstm_fslsm_results.csv", index=False)
print("Saved comparison table to data/results/student1_lstm_fslsm_results.csv")

                                                       Model                     Fusion Approach                                                                                                                            Input Features  Accuracy  Precision   Recall       F1      AUC
                           Baseline BiLSTM (sequential only)            N/A (no static features)                                                                                     Sequential (source/event/category tokens + time gaps)  0.844653   0.874784 0.938123 0.905347 0.832342
BiLSTM + FSLSM all-dimension static features (concat fusion) Concatenation (intermediate fusion) Sequential + static FSLSM all-dimension features (Input, Processing, Perception, Understanding: S-score + binary label each, 8-dim total)  0.842338   0.867276 0.945633 0.904761 0.831969

AUC change from adding all four FSLSM static dimensions: -0.0004
Saved comparison table to data/results/student1_lstm_fslsm_results.csv
